

## Objective and result

This mini project adapts the course's ants-and-bees example to three labels: **Aventurine**, **Kafka**, and **Phainon/Khaslana**. The last label intentionally includes both appearances. A pretrained ResNet18 is adapted to a small, curated dataset rather than trained from scratch.

After **three epochs of feature extraction**, the model classified **38 of 49 validation images correctly (77.6%)**. The notebook contains the analysis code. Recorded training metrics, confusion matrices, and fresh-image predictions are available in `outputs/`. This is a small course experiment, not a production recognition system.

## Dataset and preparation

The original collection contained 270 images. A visual screening pass and duplicate checks excluded 29 examples, such as reference sheets, collages, partial cutouts, mixed-character scenes, and one exact duplicate. This left **241 images**:

| Label | Training | Validation | Total |
|---|---:|---:|---:|
| aventurine | 55 | 14 | 69 |
| kafka | 58 | 15 | 73 |
| phainon_khaslana | 79 | 20 | 99 |
| **Total** | **192** | **49** | **241** |

Images were organized as `dataset/train/<class>/` and `dataset/val/<class>/`. Accepted images were converted to RGB PNG with orientation corrected. Related shots and perceptually similar images were grouped before the approximately 80/20 split; no assigned group or exact decoded-image duplicate crosses the split. The original downloads were preserved. Original URLs/videos were not supplied for the collected training images, so complete source provenance and video independence cannot be established.

The class sizes are modestly unequal and the collection mixes 3D screenshots and 2D artwork. Per-class results are therefore shown alongside overall accuracy. Augmentation creates variations of existing training images; it does not add independent source examples.

## Why transfer learning?

ResNet18 has learned visual features from ImageNet. With only 192 training images, using these pretrained features is a practical starting point. Its original 1,000-output classifier is replaced with a linear layer containing **three outputs**. Feature extraction freezes the existing parameters and trains the new classifier (1,539 parameters). The teacher's training loop calls `model.train()`, so BatchNorm running statistics still update even though backbone parameters are frozen.

## 1. Setup and project settings

Use the project's `.venv` Jupyter kernel and run from the project folder. Recorded results are available in `outputs/` without rerunning training. Running the training cell starts a fresh model and the save cell replaces the local checkpoint, so preserve the recorded checkpoint before starting another experiment.

Recorded settings: seed **42**, **3 epochs**, **Adam**, learning rate **0.001**, cross-entropy loss, training batch size **16**, validation batch size **32**, NVIDIA RTX 4060. The installed versions are in `requirements-lock.txt`; see README for setup.

The three-class baseline uses feature extraction for three epochs. In Section 11, I also compare feature extraction and full fine-tuning on an Aventurine/Kafka subset. These are different classification tasks, so their accuracy scores are reported separately.


In [ ]:
from pathlib import Path
import io
import os
import random
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import torch
import torch.nn as nn
from torchvision import datasets, transforms, models
from sklearn.metrics import ConfusionMatrixDisplay

PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / "dataset"
STRATEGY = "feature_extraction"  # Optional later experiment: "fine_tuning"
EPOCHS = 3
SEED = 42
MODEL_PATH = PROJECT_DIR / "hsr_three_class_resnet18.pt"
assert STRATEGY in {"feature_extraction", "fine_tuning"}
assert EPOCHS > 0

os.environ["TORCH_HOME"] = str(PROJECT_DIR / ".tmp" / "torch")
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Running on:", device)
if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
plt.rcParams["figure.figsize"] = (8, 4.5)
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

## 2. Load and inspect the dataset

Folder names define the labels. Training and validation must have identical class mappings. Related frames and duplicates must stay in one split; augmentation does not create independent source images.

The teacher's preprocessing is retained, with a narrower crop range (`scale=(0.7, 1.0)`) to reduce the chance of removing the character. Validation uses the pretrained weights' standard transform. Crops can still hide distinguishing details, so inspect the preview.

In [ ]:
weights = models.ResNet18_Weights.DEFAULT
validation_transform = weights.transforms()
training_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    validation_transform,
])
assert (DATA_DIR / "train").is_dir(), f"Missing training folder: {DATA_DIR / 'train'}"
assert (DATA_DIR / "val").is_dir(), f"Missing validation folder: {DATA_DIR / 'val'}"
train_dataset = datasets.ImageFolder(DATA_DIR / "train", transform=training_transform)
val_dataset = datasets.ImageFolder(DATA_DIR / "val", transform=validation_transform)
CLASSES = train_dataset.classes
assert CLASSES == ["aventurine", "kafka", "phainon_khaslana"], CLASSES
assert train_dataset.class_to_idx == val_dataset.class_to_idx
assert len(train_dataset) > 0 and len(val_dataset) > 0
counts = pd.DataFrame({
    "class": CLASSES,
    "train": np.bincount(train_dataset.targets, minlength=len(CLASSES)),
    "validation": np.bincount(val_dataset.targets, minlength=len(CLASSES)),
})
assert (counts[["train", "validation"]] > 0).all().all()
display(counts)
print("Class mapping:", train_dataset.class_to_idx)
train_loader = torch.utils.data.DataLoader(
    train_dataset, batch_size=16, shuffle=True,
    generator=torch.Generator().manual_seed(SEED), num_workers=0,
)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=0)

def show_tensor(x, ax, title=""):
    image = x.cpu().permute(1, 2, 0).numpy() * STD + MEAN
    ax.imshow(np.clip(image, 0, 1))
    ax.set_title(title, fontsize=9)
    ax.axis("off")

fig, axes = plt.subplots(3, 3, figsize=(10, 9))
for label, class_name in enumerate(CLASSES):
    indices = [i for i, target in enumerate(train_dataset.targets) if target == label][:3]
    for ax in axes[label]:
        ax.axis("off")
    for ax, index in zip(axes[label], indices):
        image, _ = train_dataset[index]
        show_tensor(image, ax, class_name)
plt.tight_layout()
plt.show()

## 3. Training and evaluation helpers

These preserve the teacher's Adam and cross-entropy approach. For each training batch, the model produces three logits, cross-entropy compares them with the labels, backpropagation computes gradients, and Adam updates the trainable parameters. Validation runs in evaluation mode without gradient tracking and never updates the model.

The helper records training and validation loss and accuracy after every epoch, extending the teacher's printed metrics so the learning curves can be inspected. Apart from recording history and narrowing the training crop range, the main approach is retained.


In [ ]:
@torch.no_grad()
def evaluate(model, loader):
    model.to(device).eval()
    loss_fn = nn.CrossEntropyLoss()
    loss_sum = 0.0
    y_true, y_pred = [], []
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        loss_sum += loss_fn(logits, y).item() * len(y)
        y_true.extend(y.cpu().tolist())
        y_pred.extend(logits.argmax(1).cpu().tolist())
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    return {
        "loss": loss_sum / len(y_true),
        "accuracy": float((y_true == y_pred).mean()),
        "y_true": y_true,
        "y_pred": y_pred,
    }

def train_model(model, train_loader, epochs=3, lr=1e-3, val_loader=None):
    model.to(device)
    optimizer = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    history = []
    for epoch in range(1, epochs + 1):
        model.train()
        total = correct = 0
        loss_sum = 0.0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            logits = model(x)
            loss = loss_fn(logits, y)
            loss.backward()
            optimizer.step()
            loss_sum += loss.item() * len(y)
            correct += (logits.argmax(1) == y).sum().item()
            total += len(y)
        row = {"epoch": epoch, "train_loss": loss_sum / total, "train_accuracy": correct / total}
        if val_loader is not None:
            validation = evaluate(model, val_loader)
            row.update(val_loss=validation["loss"], val_accuracy=validation["accuracy"])
        history.append(row)
        print(" | ".join(f"{key}: {value:.4f}" for key, value in row.items()))
    return model, pd.DataFrame(history)

## 4. Adapt ResNet18 and train

Feature extraction freezes the existing parameters and replaces `fc` with three outputs. As in the teacher's loop, `model.train()` also updates BatchNorm running statistics. Optional full fine-tuning uses a smaller learning rate.

Running the next cell starts training. Re-running it starts a fresh model, rather than continuing the previous training.

In [ ]:
torch.manual_seed(SEED)
project_model = models.resnet18(weights=weights)
for parameter in project_model.parameters():
    parameter.requires_grad = (STRATEGY == "fine_tuning")
project_model.fc = nn.Linear(project_model.fc.in_features, len(CLASSES))
trainable = sum(p.numel() for p in project_model.parameters() if p.requires_grad)
print(f"Trainable parameters: {trainable:,}")
learning_rate = 1e-3 if STRATEGY == "feature_extraction" else 1e-4
project_model, history = train_model(
    project_model, train_loader, epochs=EPOCHS, lr=learning_rate, val_loader=val_loader
)
display(history)

## 5. Learning curves and validation results

| Epoch | Training loss | Training accuracy | Validation loss | Validation accuracy |
|---|---:|---:|---:|---:|
| 1 | 1.0584 | 46.35% | 0.9986 | 51.02% |
| 2 | 0.8353 | 64.58% | 0.8183 | 53.06% |
| 3 | 0.6767 | 76.04% | 0.6812 | 77.55% |

Both losses decreased across the three epochs, and accuracy improved. This suggests that the new classifier learned useful distinctions during this run. Three epochs and one small split do not establish optimal settings or rule out overfitting. Training metrics use augmented images, whereas validation uses fixed preprocessing, so they are measured under different conditions.

The final **77.6% validation accuracy** represents 38 correct predictions and 11 mistakes. One validation mistake changes the score by about **2.04 percentage points**. It is not an independent test-set score.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for column, label in [("train_loss", "Train"), ("val_loss", "Validation")]:
    axes[0].plot(history["epoch"], history[column], marker="o", label=label)
for column, label in [("train_accuracy", "Train"), ("val_accuracy", "Validation")]:
    axes[1].plot(history["epoch"], history[column], marker="o", label=label)
axes[0].set(title="Cross-entropy loss", xlabel="Epoch", ylabel="Loss")
axes[1].set(title="Classification accuracy", xlabel="Epoch", ylabel="Accuracy", ylim=(0, 1))
for ax in axes:
    ax.set_xticks(history["epoch"])
    ax.legend()
plt.tight_layout()
plt.show()

validation = evaluate(project_model, val_loader)
print(f"Validation accuracy: {validation['accuracy']:.1%}")
print(f"Validation loss: {validation['loss']:.4f}")
print(f"One validation mistake changes accuracy by {100 / len(val_dataset):.2f} percentage points.")
ConfusionMatrixDisplay.from_predictions(
    validation["y_true"], validation["y_pred"], labels=list(range(len(CLASSES))),
    display_labels=CLASSES, cmap="Blues", colorbar=False,
)
plt.xticks(rotation=20, ha="right")
plt.xlabel("Predicted class")
plt.ylabel("True class")
plt.tight_layout()
plt.show()

## 6. Inspect mistakes

Look for unusual poses, heavy crops, hidden faces, mixed-character images, background shortcuts, and differences between 2D art and 3D screenshots. Zero errors on this small validation set would not prove perfect generalization.

In [ ]:
incorrect = np.flatnonzero(validation["y_true"] != validation["y_pred"])[:6]
if len(incorrect):
    fig, axes = plt.subplots(1, len(incorrect), figsize=(3.5 * len(incorrect), 3.5), squeeze=False)
    for ax, index in zip(axes[0], incorrect):
        image, target = val_dataset[index]
        predicted = validation["y_pred"][index]
        show_tensor(image, ax, f"True: {CLASSES[target]}\nPredicted: {CLASSES[predicted]}")
    plt.tight_layout()
    plt.show()
else:
    print("No incorrect predictions in this validation split.")

## 7. Save, reload, and predict fresh images

The checkpoint stores the class order with the weights. Predictions use validation preprocessing. Only load checkpoints you trust. Softmax scores are not calibrated confidence, and the model always chooses one of its three classes, even for unrelated images.

In [ ]:
torch.save({
    "weights": project_model.state_dict(), "classes": CLASSES,
    "architecture": "resnet18", "strategy": STRATEGY,
}, MODEL_PATH)
checkpoint = torch.load(MODEL_PATH, map_location=device, weights_only=True)
loaded_model = models.resnet18(weights=None)
loaded_model.fc = nn.Linear(loaded_model.fc.in_features, len(checkpoint["classes"]))
loaded_model.load_state_dict(checkpoint["weights"])
loaded_model = loaded_model.to(device).eval()
print("Saved and reloaded:", MODEL_PATH)

# The same preprocessing must be used at validation and inference.
def predict(source):
    if str(source).startswith(("https://", "http://")):
        response = requests.get(source, timeout=30)
        response.raise_for_status()
        image = Image.open(io.BytesIO(response.content)).convert("RGB")
    else:
        with Image.open(source) as original:
            image = original.convert("RGB")
    x = validation_transform(image).unsqueeze(0).to(device)
    with torch.no_grad():
        scores = loaded_model(x).softmax(1)[0].cpu()
    class_names = checkpoint["classes"]
    result = pd.DataFrame({"class": class_names, "softmax_score": scores.numpy()})
    plt.figure(figsize=(5, 5))
    plt.imshow(image)
    plt.axis("off")
    plt.title(f"Predicted: {class_names[int(scores.argmax())]}")
    plt.show()
    display(result)
    return class_names[int(scores.argmax())]

# Uncomment after selecting a genuinely new image outside train/val:
# predict("path/to/new_image.jpg")

## 8. What the baseline shows

Kafka had the highest recall in this split (**14/15**), followed by Phainon/Khaslana (**15/20**) and Aventurine (**9/14**). These fractions describe the proportion correctly recognized within each true class. Ten of the eleven errors were predictions of Kafka for another class.

The displayed mistakes include unusual views, distant subjects, and screenshots containing substantial interface/background content. The complete set of errors was also reviewed. These characteristics suggest possible difficulties, but this experiment does not isolate their causes. Cropping, style, lighting, and background shortcuts may all affect predictions.

The next section checks new illustrations and an unrelated character using the saved model, without retraining.


## 9. Fresh-image demonstration

These four images were downloaded from the [StarRailRes game-resource archive](https://github.com/Mar-7th/StarRailRes), pinned to the commit recorded in `outputs/fresh-images/source-manifest.json`. They were visually reviewed and checked for exact and perceptual similarity against train/validation. A repeated Aventurine illustration and Kafka splash-art variant were rejected. The remaining images are new inputs to this model, not proof of source independence for the original dataset.

This uses the saved baseline checkpoint and exactly its validation preprocessing. No training occurs. The examples are 2D artwork, whereas much of training is 3D imagery. Four chosen examples are a demonstration, not an unbiased test set or a new accuracy estimate. Game artwork remains its owner's material; these files stay local.


In [ ]:
import json
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import torch
import torch.nn as nn
from torchvision import models

fresh_dir = Path.cwd() / "outputs" / "fresh-images"
saved = torch.load(Path.cwd() / "hsr_three_class_resnet18.pt", map_location="cpu", weights_only=True)
fresh_classes = saved["classes"]
assert fresh_classes == ["aventurine", "kafka", "phainon_khaslana"]
inference_device = "cuda" if torch.cuda.is_available() else "cpu"
inference_model = models.resnet18(weights=None)
inference_model.fc = nn.Linear(inference_model.fc.in_features, len(fresh_classes))
inference_model.load_state_dict(saved["weights"])
inference_model.to(inference_device).eval()
inference_transform = models.ResNet18_Weights.DEFAULT.transforms()
sources = json.loads((fresh_dir / "source-manifest.json").read_text())
selected = [item for item in sources if item["selected"]]
rows = []
fig, axes = plt.subplots(1, len(selected), figsize=(16, 5))
for ax, item in zip(axes, selected):
    with Image.open(fresh_dir / item["file"]) as original:
        image = original.convert("RGB")  # Same RGB loading as the saved pipeline
    batch = inference_transform(image).unsqueeze(0).to(inference_device)
    with torch.no_grad():
        scores = inference_model(batch).softmax(1)[0].cpu().numpy()
    predicted = fresh_classes[int(scores.argmax())]
    row = {"file": item["file"], "expected_class": item["expected_class"], "predicted_class": predicted}
    row.update({name + "_score": float(value) for name, value in zip(fresh_classes, scores)})
    rows.append(row)
    ax.imshow(image)
    ax.set_title(f"Actual: {item['expected_class']}\nPredicted: {predicted}\nTop softmax: {scores.max():.1%}", fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.savefig(fresh_dir / "fresh-predictions.png", dpi=130)
plt.show()
fresh_results = pd.DataFrame(rows)
assert len(fresh_results) == 4
assert np.isfinite(fresh_results.filter(like="_score").to_numpy()).all()
assert np.allclose(fresh_results.filter(like="_score").sum(axis=1), 1)
fresh_results.to_csv(fresh_dir / "predictions.csv", index=False)
display(fresh_results)
print("Himeko is outside the trained labels: a predicted label is forced, not a valid recognition.")


### Observations from the recorded experiments

The baseline validation score was **38/49 (77.6%)**. Per-class recall was Aventurine **9/14**, Kafka **14/15**, and combined Phainon/Khaslana **15/20**. Ten of the eleven errors were another class predicted as Kafka. Both training and validation loss decreased across three epochs; the short run alone cannot establish whether more epochs would improve unseen-image performance.

All eleven validation mistakes were reviewed. They include a rear-view Aventurine, distant/full-body subjects, UI-heavy shots, and Phainon images under strong orange or purple lighting. These observations suggest possible difficulties, not proven causes. Validation center cropping can also hide parts of a wide screenshot. No settings or images were changed in response.

| New input | Expected | Predicted | Top softmax score |
|---|---|---|---:|
| Aventurine splash illustration | aventurine | phainon_khaslana | 39.8% |
| Kafka: Patience Is All You Need | kafka | kafka | 40.1% |
| Phainon: Thus Burns the Dawn | phainon_khaslana | phainon_khaslana | 62.3% |
| Himeko: Night on the Milky Way | outside trained classes | phainon_khaslana | 50.4% |

Kafka and Phainon were correctly recognized in these examples; Aventurine was not. This selection is too small to provide a reliable new accuracy estimate. Phainon's example does not independently test both appearances in the combined class. Himeko receives a label because softmax always distributes scores over the three known outputs; its 50.4% score does not make it Phainon or demonstrate calibrated confidence.

These 2D illustrations differ from many 3D training screenshots. The Aventurine asset has transparency, and converting it to RGB with the existing pipeline exposes background texture stored in transparent pixels. This is another input-format limitation; the baseline preprocessing was deliberately preserved, so this example does not isolate character-recognition skill from image-format effects. Any preprocessing change belongs in a separate experiment.

Source URLs, commit, hashes, and similarity checks for the selected images are in `outputs/fresh-images/source-manifest.json`; predictions are in `predictions.csv` there. No model retraining or dataset split changes were made.


## 10. Three-class baseline conclusion

The course's ResNet18 workflow could be reused for a three-class character dataset by changing the data folders and final layer. The small feature-extraction baseline reached **38/49 correct validation predictions** after three epochs. Its saved checkpoint reproduced the same outputs after reloading.

The fresh-image demonstration recognized Kafka and Phainon but misclassified Aventurine. Himeko, an unrelated character, was also assigned a trained label because the model has no unknown-class option. Softmax scores should not be interpreted as calibrated confidence.

The main limitations are the small and somewhat imbalanced dataset, mixed illustration/screenshot styles, incomplete original-source provenance, possible related-source leakage, a single small validation split, and no independent test set. The fresh illustrations are a deliberately small demonstration, not a generalization benchmark. The Phainon example also does not separately test both appearances in the combined label. Transparent image handling and center cropping can introduce additional input difficulties.

Section 11 follows up with a two-class experiment comparing additional epochs and full fine-tuning on the same Aventurine/Kafka split.

### References and reproducibility

- Course reference: the teacher's Session 19 transfer-learning notebook (ants and bees).
- Dataset: locally collected images. Complete original URLs were not available; preparation and limitations are described above.
- Fresh artwork: [StarRailRes](https://github.com/Mar-7th/StarRailRes), commit `dbe8cdfcfb0bf657b9fe1cf92d5537f118ccb487`. Exact image URLs and hashes for the selected inputs are in `outputs/fresh-images/source-manifest.json`. Game artwork remains the property of its respective owners.
- Recorded settings/results: `outputs/baseline/results.json` and `outputs/baseline/history.csv`.
- Fresh predictions: `outputs/fresh-images/predictions.csv`.
- Environment and run instructions: `README.md` and `requirements-lock.txt`.

This notebook is organized around the supplied course example. No separate grading rubric or submission-format requirements were provided.


## 11. Two-class model and unseen characters

The earlier three-class experiment remains an earlier result. Here the training classes are **Aventurine and Kafka only**. Phainon/Khaslana is now unseen by these models. Existing files and train/validation membership are unchanged: subsets are filtered in memory. All runs start from ImageNet weights with seed 42 and the same preprocessing, batch sizes, Adam, and cross-entropy helpers used above.

We compare feature extraction for 3 and 10 epochs (learning rate 0.001), and whole-network fine-tuning for 10 epochs (learning rate 0.0001). Each is a fresh run rather than continuation of another model. Feature extraction retains the teacher's BatchNorm behavior. Timing includes training plus each epoch's validation, not initial model loading. The final epoch of the run with highest validation accuracy is selected; ties prefer the earlier listed run. This selection uses a small validation set and makes its reported performance optimistic. There is no independent test set.


In [ ]:
import copy
import time
import json
import gc

TWO_CLASSES = ["aventurine", "kafka"]
TWO_OUT = PROJECT_DIR / "outputs" / "two_class"
TWO_OUT.mkdir(parents=True, exist_ok=True)

def two_class_subset(source_dataset):
    # Keep original image paths and transforms, remapping only the two labels.
    subset = copy.copy(source_dataset)
    subset.samples = [(path, TWO_CLASSES.index(source_dataset.classes[label]))
                      for path, label in source_dataset.samples
                      if source_dataset.classes[label] in TWO_CLASSES]
    subset.imgs = subset.samples
    subset.targets = [label for _, label in subset.samples]
    subset.classes = TWO_CLASSES.copy()
    subset.class_to_idx = {name: i for i, name in enumerate(TWO_CLASSES)}
    return subset

two_train = two_class_subset(train_dataset)
two_val = two_class_subset(val_dataset)
assert len(two_train) == 113 and len(two_val) == 29
assert two_train.class_to_idx == two_val.class_to_idx
assert not ({p for p, _ in two_train.samples} & {p for p, _ in two_val.samples})
two_counts = pd.DataFrame({"class": TWO_CLASSES,
    "train": np.bincount(two_train.targets, minlength=2),
    "validation": np.bincount(two_val.targets, minlength=2)})
display(two_counts)
print(f"One validation mistake = {100 / len(two_val):.2f} percentage points.")


In [ ]:
# Same helpers as the baseline; reset randomness and loaders for each fresh run.
run_specs = [("feature_extraction_3", "feature_extraction", 3, 1e-3),
             ("feature_extraction_10", "feature_extraction", 10, 1e-3),
             ("fine_tuning_10", "fine_tuning", 10, 1e-4)]
run_rows, run_histories = [], {}
best_accuracy = -1.0
best_state = None
# Release previous GPU models; their saved original checkpoint is retained.
for old_name in ["project_model", "loaded_model", "inference_model"]:
    if old_name in globals():
        globals()[old_name].cpu()
gc.collect()
if device == "cuda": torch.cuda.empty_cache()

for run_name, strategy, epochs, lr in run_specs:
    random.seed(42); np.random.seed(42); torch.manual_seed(42)
    if device == "cuda": torch.cuda.manual_seed_all(42)
    loader_train = torch.utils.data.DataLoader(two_train, batch_size=16, shuffle=True,
        generator=torch.Generator().manual_seed(42), num_workers=0)
    loader_val = torch.utils.data.DataLoader(two_val, batch_size=32, shuffle=False, num_workers=0)
    candidate = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    for parameter in candidate.parameters():
        parameter.requires_grad = strategy == "fine_tuning"
    candidate.fc = nn.Linear(candidate.fc.in_features, 2)
    trainable_count = sum(p.numel() for p in candidate.parameters() if p.requires_grad)
    if device == "cuda": torch.cuda.synchronize()
    started = time.perf_counter()
    candidate, run_history = train_model(candidate, loader_train, epochs=epochs, lr=lr, val_loader=loader_val)
    if device == "cuda": torch.cuda.synchronize()
    seconds = time.perf_counter() - started
    measured = evaluate(candidate, loader_val)
    final = run_history.iloc[-1]
    row = {"run": run_name, "strategy": strategy, "epochs": epochs, "lr": lr,
        "train_loss": float(final.train_loss), "train_accuracy": float(final.train_accuracy),
        "val_loss": measured["loss"], "val_accuracy": measured["accuracy"],
        "trainable_parameters": trainable_count, "training_seconds": seconds}
    assert np.isfinite(run_history.to_numpy()).all()
    assert abs(float(final.val_accuracy)-measured["accuracy"]) < 1e-12
    run_rows.append(row); run_histories[run_name] = run_history
    run_history.to_csv(TWO_OUT / f"history_{run_name}.csv", index=False)
    if measured["accuracy"] > best_accuracy:
        best_accuracy = measured["accuracy"]
        best_name = run_name
        best_state = {key: value.detach().cpu().clone() for key, value in candidate.state_dict().items()}
    del candidate
    gc.collect()
    if device == "cuda": torch.cuda.empty_cache()

comparison_two = pd.DataFrame(run_rows)
comparison_two.to_csv(TWO_OUT / "comparison.csv", index=False)
display(comparison_two)
print("Selected run:", best_name)
best_two = models.resnet18(weights=None)
best_two.fc = nn.Linear(best_two.fc.in_features, 2)
best_two.load_state_dict(best_state)
best_two.to(device).eval()
best_validation = evaluate(best_two, loader_val)
assert best_validation["accuracy"] == best_accuracy
torch.save({"weights": best_state, "classes": TWO_CLASSES, "run": best_name,
            "architecture": "resnet18", "seed": 42}, PROJECT_DIR / "hsr_two_class_resnet18.pt")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for name, frame in run_histories.items():
    axes[0].plot(frame.epoch, frame.train_loss, label=name + " train")
    axes[0].plot(frame.epoch, frame.val_loss, linestyle="--", label=name + " val")
    axes[1].plot(frame.epoch, frame.train_accuracy, label=name + " train")
    axes[1].plot(frame.epoch, frame.val_accuracy, linestyle="--", label=name + " val")
axes[0].set(title="Two-class cross-entropy loss", xlabel="Epoch", ylabel="Loss")
axes[1].set(title="Two-class accuracy", xlabel="Epoch", ylabel="Accuracy", ylim=(0,1.05))
for ax in axes: ax.legend(fontsize=7)
plt.tight_layout(); plt.savefig(TWO_OUT / "learning_curves.png", dpi=130); plt.show()
ConfusionMatrixDisplay.from_predictions(best_validation["y_true"], best_validation["y_pred"],
    labels=[0,1], display_labels=TWO_CLASSES, cmap="Blues", colorbar=False)
plt.title(best_name + " — validation")
plt.tight_layout(); plt.savefig(TWO_OUT / "confusion_matrix.png", dpi=130); plt.show()
wrong_two = np.flatnonzero(best_validation["y_true"] != best_validation["y_pred"])
if len(wrong_two):
    cols = min(4, len(wrong_two)); rows = int(np.ceil(len(wrong_two)/cols))
    fig, axes = plt.subplots(rows, cols, figsize=(4*cols, 4*rows), squeeze=False)
    for ax in axes.flat: ax.axis("off")
    for ax, index in zip(axes.flat, wrong_two):
        x, target = two_val[int(index)]
        predicted = best_validation["y_pred"][index]
        show_tensor(x, ax, f"True: {TWO_CLASSES[target]}\nPredicted: {TWO_CLASSES[predicted]}")
    plt.tight_layout(); plt.savefig(TWO_OUT / "validation_mistakes.png", dpi=130); plt.show()
else:
    print("No mistakes in this small selected validation split; this does not prove perfect generalization.")


### 11.1 Reaction to characters absent from two-class training

All Phainon/Khaslana images from both existing folders are held out of this model's training. Optional folders under `outputs/unseen/` are evaluated if present; missing or empty folders are reported. The four existing fresh images are reported separately: Kafka/Aventurine remain known identities, while Phainon and Himeko are unseen identities. No images are copied or moved.

Inference uses the original validation transform and `convert("RGB")`. Transparent-PNG artifacts are therefore retained as a known limitation. Counts and softmax-score summaries describe the selected model's outputs; they do not establish calibrated confidence or unknown-character detection.


In [ ]:
entries = []
for path, target in two_val.samples:
    entries.append({"file": path, "group": "validation", "expected": TWO_CLASSES[target], "unseen": False})
phainon_paths = [path for dataset in [train_dataset, val_dataset]
                 for path, label in dataset.samples if dataset.classes[label] == "phainon_khaslana"]
assert len(phainon_paths) == 99
for path in phainon_paths:
    entries.append({"file": path, "group": "unseen/phainon_khaslana", "expected": "phainon_khaslana", "unseen": True})
extra_counts = {}
unseen_root = PROJECT_DIR / "outputs" / "unseen"
extensions = {".png", ".jpg", ".jpeg", ".webp", ".bmp"}
if not unseen_root.is_dir(): print("No optional outputs/unseen folder; skipped.")
else:
    for folder in sorted(p for p in unseen_root.iterdir() if p.is_dir()):
        files = sorted(p for p in folder.rglob("*") if p.suffix.lower() in extensions)
        extra_counts[folder.name] = len(files)
        print(f"Optional folder {folder.name}: {len(files)} images")
        if folder.name.lower() in TWO_CLASSES:
            print("Known training identity; this folder is skipped as an unseen group.")
            continue
        for path in files:
            entries.append({"file": str(path), "group": "unseen/"+folder.name,
                            "expected": folder.name, "unseen": True})
manifest = json.loads((PROJECT_DIR / "outputs/fresh-images/source-manifest.json").read_text())
fresh_selected = [r for r in manifest if r.get("selected", True)]
assert len(fresh_selected) == 4
for item in fresh_selected:
    identity = "himeko" if item["expected_class"] == "out_of_class" else item["expected_class"]
    entries.append({"file": str(PROJECT_DIR / "outputs/fresh-images" / item["file"]),
        "group": "fresh/"+identity, "expected": identity, "unseen": identity not in TWO_CLASSES})

prediction_rows = []
for start in range(0, len(entries), 32):
    part = entries[start:start+32]
    tensors = []
    for entry in part:
        with Image.open(entry["file"]) as raw:
            tensors.append(validation_transform(raw.convert("RGB")))
    with torch.no_grad():
        scores = best_two(torch.stack(tensors).to(device)).softmax(1).cpu().numpy()
    for entry, score in zip(part, scores):
        prediction_rows.append({**entry, "predicted": TWO_CLASSES[int(score.argmax())],
            "aventurine_score": float(score[0]), "kafka_score": float(score[1]),
            "top_softmax": float(score.max()),
            "correct": TWO_CLASSES[int(score.argmax())] == entry["expected"] if not entry["unseen"] else None})
two_predictions = pd.DataFrame(prediction_rows)
two_predictions["file"] = two_predictions["file"].map(lambda p: str(Path(p).relative_to(PROJECT_DIR)))
two_predictions.to_csv(TWO_OUT / "predictions.csv", index=False)
summaries = []
for group, frame in two_predictions.groupby("group", sort=True):
    counts_assigned = frame.predicted.value_counts()
    row = {"group": group, "count": len(frame), "mean_top_softmax": float(frame.top_softmax.mean()),
           "median_top_softmax": float(frame.top_softmax.median())}
    for name in TWO_CLASSES:
        row[name+"_count"] = int(counts_assigned.get(name, 0))
        row[name+"_percent"] = 100*row[name+"_count"]/len(frame)
    summaries.append(row)
unseen_summary = pd.DataFrame(summaries)
unseen_summary.to_csv(TWO_OUT / "group_summary.csv", index=False)
display(unseen_summary)
validation_scores = two_predictions[two_predictions.group == "validation"]
confidence_rows = []
for label, mask in [("correct validation", validation_scores.correct == True),
                    ("incorrect validation", validation_scores.correct == False)]:
    values = validation_scores.loc[mask, "top_softmax"]
    confidence_rows.append({"subset": label, "count": len(values),
        "mean_top_softmax": float(values.mean()) if len(values) else None,
        "median_top_softmax": float(values.median()) if len(values) else None})
confidence_comparison = pd.DataFrame(confidence_rows)
display(confidence_comparison)
confidence_comparison.to_csv(TWO_OUT / "validation_score_summary.csv", index=False)
unseen_scores = two_predictions.loc[two_predictions.unseen, "top_softmax"]
bins = np.linspace(.5, 1, 11)
plt.figure(figsize=(9,4))
plt.hist(validation_scores.top_softmax, bins=bins, alpha=.6, label=f"Known validation (n={len(validation_scores)})")
plt.hist(unseen_scores, bins=bins, alpha=.6, label=f"Unseen identity images (n={len(unseen_scores)})")
plt.xlabel("Top softmax score (not calibrated confidence)"); plt.ylabel("Image count")
plt.title("Known vs unseen images — unequal group sizes, raw counts")
plt.legend(); plt.tight_layout(); plt.savefig(TWO_OUT / "softmax_histogram.png", dpi=130); plt.show()

from sklearn.metrics import confusion_matrix
two_result = {"classes": TWO_CLASSES, "seed":42, "batch_sizes":{"train":16,"validation":32},
    "optimizer":"Adam", "loss":"CrossEntropyLoss", "train_transform":repr(training_transform),
    "val_transform":repr(validation_transform), "counts":two_counts.to_dict(orient="records"),
    "runs":run_rows, "selection":"Highest final-epoch validation accuracy; ties prefer earlier listed run; optimistic selection on small validation set.",
    "best_run":best_name, "best_validation_accuracy":best_accuracy,
    "confusion_matrix":confusion_matrix(best_validation["y_true"],best_validation["y_pred"],labels=[0,1]).tolist(),
    "unseen_groups":summaries, "validation_score_groups":confidence_rows,
    "optional_folder_counts":extra_counts, "python":__import__("platform").python_version(),
    "torch":torch.__version__, "torchvision":__import__("torchvision").__version__,
    "device":device, "gpu":torch.cuda.get_device_name(0) if device=="cuda" else None}
(TWO_OUT / "results.json").write_text(json.dumps(two_result, indent=2), encoding="utf-8")
reloaded_two = models.resnet18(weights=None); reloaded_two.fc = nn.Linear(reloaded_two.fc.in_features,2)
saved_two = torch.load(PROJECT_DIR / "hsr_two_class_resnet18.pt", map_location=device, weights_only=True)
assert saved_two["classes"] == TWO_CLASSES
reloaded_two.load_state_dict(saved_two["weights"]); reloaded_two.to(device).eval()
check_x, _ = next(iter(loader_val))
with torch.no_grad():
    torch.testing.assert_close(best_two(check_x.to(device)), reloaded_two(check_x.to(device)), rtol=0, atol=0)
assert len(validation_scores)==29
assert sum(row["count"] for row in summaries)==len(two_predictions)
assert two_predictions[["aventurine_score","kafka_score"]].sum(axis=1).sub(1).abs().max()<1e-5
assert not ({str(Path(p).relative_to(PROJECT_DIR)) for p,_ in two_train.samples} & set(two_predictions.loc[two_predictions.unseen,"file"]))
print("PASS: 29 validation cases, score sums, unseen-path separation, and identical checkpoint reload logits.")
print("Final comparison:")
display(comparison_two)


In [ ]:
# Generate conclusions from measured values only.
from IPython.display import Markdown, display
correct_two = int(np.trace(two_result["confusion_matrix"]))
text = f"""### 11.2 What this experiment shows

The selected run was **{best_name}**, with **{correct_two}/29** correct validation predictions (**{best_accuracy:.1%}**). One error changes validation accuracy by **{100/29:.2f} percentage points**. Selection among three runs on this same small set is optimistic; there is no independent test set.
"""
for row in summaries:
    if row["group"] != "validation":
        text += (f"\n- **{row['group']}**: {row['count']} images; "
            f"Aventurine {row['aventurine_count']} ({row['aventurine_percent']:.1f}%), "
            f"Kafka {row['kafka_count']} ({row['kafka_percent']:.1f}%); "
            f"mean/median top softmax {row['mean_top_softmax']:.3f}/{row['median_top_softmax']:.3f}.")
text += """

Softmax always picks one of the two trained classes. A high or low score is not calibrated confidence and does not establish whether an image belongs to an unknown character. The unseen groups mix 2D illustrations and 3D screenshots: differences may reflect style, framing, or lighting as well as identity. No cause is established by these observations. The original transparent-PNG RGB conversion artifact remains. Data are small, source provenance is incomplete, and no independent test set is available. The earlier three-class result is a different task and cannot be compared as though its accuracy used the same labels or validation population.
"""
display(Markdown(text))
(TWO_OUT / "conclusion.md").write_text(text, encoding="utf-8")


## 12. Overall conclusion

I adapted the course's ResNet18 example to Honkai: Star Rail characters. The three-class baseline got **38/49 validation images correct (77.6%)**. In the separate two-class experiment, ten-epoch fine-tuning got **28/29 (96.6%)**, compared with **27/29 (93.1%)** for ten-epoch feature extraction.

Fine-tuning scored slightly higher on this small validation set—just one extra correct image. There was no independent test set, so this does not establish that it will consistently perform better. The unseen-character examples also showed that the model always picks a known label, even when the character is outside its training classes.
